# Fig 2D / S3A — LD sketch vs. full LD (data processing)

Per-region error of LD from the genotype sketch vs. LD from a random subsample of individuals, across sketch dimension B (UK Biobank).

**Output:** `figure_data/Fig2D.LD_Sketch_MSE.rds` (+ `.csv`) — used by Fig 2D (MSE) and Fig S3A (max |Δr|, concordance).

### LD mismatch metrics

All metrics compare a reference LD matrix (**sketch** = `U Uᵀ/B`; **subset** = dense LD from a random sub-sample of individuals) against the **in-sample** LD (`LD_full`), computed on the shared variant set in identical order. Only **upper-triangular off-diagonal** pairs are used, restricted to informative LD (`|R_insample| > ld_thresh`, default 0.05); references with fewer than `min_pairs` kept pairs return `NA`.

- **`mse`** — Mean squared error between the reference and in-sample correlations over kept pairs: `mean((r_ref − r_insample)²)`. The raw discrepancy; lower is better. For the sketch it decreases as `B` grows.
- **`concord`** — Pearson correlation between the reference and in-sample off-diagonal entries: how well the reference preserves the *shape/ranking* of the LD pattern, independent of absolute scale. Near 1 = structure faithfully reproduced.
- **`D2`** — Bias-corrected MSE: `mse − mean((1 + r_ref²)/B)`. The stochastic sketch adds Monte-Carlo variance of order `(1 + r²)/B` to each estimated correlation; subtracting it isolates *systematic* mismatch beyond sketching noise. `D2 → 0` means the sketch differs from in-sample only by expected sampling noise (no structural bias); `D2 > 0` flags real mismatch. `NA` for dense references with no `B`.
- **`n_pairs`** — Number of off-diagonal pairs retained after the `|R_insample| > ld_thresh` filter; the effective sample size behind each metric.
- **`p`** — Number of variants in the region. **`B`** — divisor used in `D2` (sketch dimension for the sketch; `NA` for the dense subset). **`B_setting`** — the sweep's `B_stoc` for this file (grouping key for the across-B panel).
- **`med_sq`** — Median of the squared pairwise differences, `median((r_ref − r_insample)²)`. Robust counterpart to `mse`: describes the *typical* pair's error, unaffected by a few large-LD outliers. A large gap between `mse` and `med_sq` means the error is driven by a small number of badly-reproduced pairs.
- **`max_abs_diff`** — Worst-case discrepancy, `max(|r_ref − r_insample|)` over kept pairs. The single most mis-estimated correlation; useful because fine-mapping can be derailed by one badly wrong off-diagonal entry even when the average error is small.

In [1]:
source("../../R/utils.R")

data_name <- "Fig2D.LD_Sketch_MSE"

result <- readRDS("/hpc/users/sunh17/haochen/rss_qc/2026_output/UKBB_sketch_dimension/ld_mismatch/ld_mismatch_combined.rds")

metrics_long <- result %>%
  filter(!is.na(mse)) %>%
  mutate(
    resource = recode(reference, sketch = "LD Sketch", subset = "Random Subset"),
    resource = factor(resource, levels = names(pal_ld_resource))
  )
stopifnot(!any(is.na(metrics_long$resource)))

save_data(metrics_long, data_name)

In [2]:
## mean sketch MSE at B = 10,000 (quoted in the text)
metrics_long %>% filter(reference == "sketch" & B == 10000) %>% pull(mse) %>% mean()

[1] 8.626058e-05